
**Detección de Riesgos Contractuales, Concentración de Proveedores y Anomalías en la Contratación Pública Colombiana**

Un análisis comparativo entre los periodos presidenciales 2018-2022 y 2022-2026 utilizando técnicas descriptivas, diagnósticas, predictivas y prescriptivas sobre SECOP II

UNIVERSIDAD EXTERNADO DE COLOMBIA
MAESTRÍA EN ANALÍTICA DE DATOS PARA CONTABILIDAD Y AUDITORÍA

PROYECTO FINAL

Diegoo Fernando SoLarte Lame

Objetivo:

Analizar la contratación pública colombiana registrada en SECOP II para identificar patrones de riesgo contractual,
concentración de proveedores, prórrogas y comportamientos atípicos mediante técnicas descriptivas, diagnósticas, predictivas y prescriptivas.

PREGUNTA DE INVESTIGACIÓN

¿Cómo ha evolucionado la contratación pública entre los periodos 2018-2022 y 2022-2026 y qué factores permiten identificar contratos con mayor riesgo de desviaciones,concentración de adjudicación y comportamientos
anómalos que deban ser priorizados por auditoría?

**LIBRERÍAS**

In [12]:
import requests
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from tqdm import tqdm

# Import for retries
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

pd.set_option("display.max_columns", None)

sns.set_theme(style="whitegrid")

In [13]:
from wordcloud import WordCloud

In [14]:
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    precision_score,
    recall_score,
    fbeta_score
)

**Carga de SECOP II**
"""
Carga de una muestra representativa de SECOP II.

Se extraen registros de dos periodos:

1. 2018-08-07 a 2022-08-06
2. 2022-08-07 a la actualidad

La extracción se realiza mediante
la API SODA de Datos Abiertos.


In [17]:
BASE_URL = "https://www.datos.gov.co/api/v3/views/jbjy-vk9h/query.json"


**Selección de columnas**

In [16]:
columnas = [
"nombre_entidad",
"nit_entidad",
"departamento",
"proveedor_adjudicado",
"documento_proveedor",
"modalidad_de_contratacion",
"valor_del_contrato",
"valor_de_pago_adelantado",
"fecha_de_firma",
"fecha_de_inicio_del_contrato",
"fecha_de_fin_del_contrato",
"duracion"
]

In [18]:
SELECT_COLS = ",".join(columnas)

**Función de descarga**

In [19]:
def descargar_periodo(where_clause,
                      limite_total=150000,
                      batch=10000,
                      retries=3,
                      backoff_factor=0.3,
                      timeout=60):

    lista = []

    # Configure retry strategy
    retry_strategy = Retry(
        total=retries,
        backoff_factor=backoff_factor,
        status_forcelist=[429, 500, 502, 503, 504],
        allowed_methods=["HEAD", "GET", "OPTIONS"]
    )
    adapter = HTTPAdapter(max_retries=retry_strategy)
    http = requests.Session()
    http.mount("https://", adapter)
    http.mount("http://", adapter)

    try:
        for offset in tqdm(
            range(0, limite_total, batch)
        ):

            query = {
                "$select": SELECT_COLS,
                "$where": where_clause,
                "$limit": batch,
                "$offset": offset
            }

            try:
                r = http.get(
                    BASE_URL,
                    params=query,
                    timeout=timeout # Add timeout to prevent hanging requests
                )
                r.raise_for_status() # Raise an exception for bad status codes
            except requests.exceptions.RequestException as e:
                print(f"Error fetching data for offset {offset}: {e}")
                # Optionally, you can log the error and decide to skip or break
                continue # Skip to the next batch on error

            temp = pd.DataFrame(r.json())

            # Reducir memoria
            for col in temp.select_dtypes(include="object"):
                temp[col] = temp[col].astype("string")

            lista.append(temp)

    except KeyboardInterrupt:
        print("Descarga interrumpida por el usuario. Retornando datos parciales.")

    return pd.concat(
        lista,
        ignore_index=True
    ) if lista else pd.DataFrame() # Return empty DataFrame if no data was collected

**Descargar Gobierno 2018-2022**

In [20]:
where_duque = """

fecha_de_firma >= '2018-08-07T00:00:00'
AND
fecha_de_firma <= '2022-08-06T23:59:59'

"""

In [ ]:
df_duque = descargar_periodo(
    where_duque,
    limite_total=150000
)

df_duque[
    "periodo_presidencial"
] = "2018-2022"

  0%|          | 0/15 [00:00<?, ?it/s]

**Descargar Gobierno 2022-2026**

In [ ]:
where_petro = """

fecha_de_firma >= '2022-08-07T00:00:00'

"""

In [ ]:
df_petro = descargar_periodo(
    where_petro,
    limite_total=150000
)

df_petro[
    "periodo_presidencial"
] = "2022-2026"

**Unir ambas muestras**

In [ ]:
df = pd.concat(
    [
        df_duque,
        df_petro
    ],
    ignore_index=True
)

In [ ]:
print(df.shape)

**Guardar respaldo**

In [ ]:
df.to_parquet(
    "secop_contratos.parquet"
)

**Verificación**

In [ ]:
print(df.shape)
df.head()
df.info()
df.sample(5)
df.describe()


**Conversión de tipos**

In [ ]:
numericas = [

"valor_del_contrato",
"valor_de_pago_adelantado",
"duracion"

]

for col in numericas:

    if col in df.columns:

        df[col] = pd.to_numeric(
            df[col],
            errors="coerce"
        )

**Fechas**

In [ ]:
fechas = [

"fecha_de_firma",

"fecha_de_inicio_del_contrato",

"fecha_de_fin_del_contrato"

]

for col in fechas:

    if col in df.columns:

        df[col] = pd.to_datetime(
            df[col],
            errors="coerce"
        )

**Primer chequeo**

In [ ]:
print(df.shape)

print()

print(df.isnull().sum().head(20))

**Calidad de Datos**

**Nulos**

In [ ]:
missing = (
    df.isna()
      .sum()
      .sort_values(ascending=False)
)

missing.head(20)

missing.head(20).plot(
    kind='bar'
)

plt.title(
    "Valores faltantes por variable"
)

**Duplicados**

In [ ]:
duplicados = df.duplicated().sum()

print(
    f"Duplicados encontrados: {duplicados:,}"
)

**Outliers**

In [ ]:
sns.boxplot(
    x=np.log1p(
        df['valor_del_contrato']
    )
)

**Nube de palabras**

In [ ]:
texto = " ".join(
    df[
       "descripcion_del_proceso"
    ]
    .dropna()
    .astype(str)
)

In [ ]:
wc = WordCloud(
    width=1500,
    height=800,
    background_color='white',
    max_words=250,
    colormap='viridis'
).generate(texto)

In [ ]:
plt.figure(figsize=(18,8))

plt.imshow(wc)

plt.axis("off")

plt.title(
    "Frecuencia de términos en los objetos contractuales"
)

plt.show()

INTERPRETACIÓN

Hallazgo: (AJUSTARRRRRRR)

Los términos más frecuentes se encuentran relacionados
con prestación de servicios, apoyo a la gestión,
desarrollo institucional y asistencia técnica.

Esto evidencia una alta participación de contratos
de servicios profesionales dentro de SECOP II.

Hallazgo:

Los términos más frecuentes se encuentran relacionados con prestación de servicios, apoyo a la gestión, desarrollo institucional y asistencia técnica.

Esto evidencia una alta participación de contratos de servicios profesionales dentro de SECOP II.

**ANALÍTICA DESCRIPTIVA**

Top 20 entidades

In [ ]:
top_ent = (

df.groupby(
    'nombre_entidad'
)

['valor_del_contrato']

.sum()

.nlargest(20)

)

In [ ]:
sns.barplot(
    x=top_ent.values,
    y=top_ent.index
)

**Top proveedores**

In [ ]:
top_prov = (

df.groupby(
    'proveedor_adjudicado'
)

['valor_del_contrato']

.sum()

.nlargest(20)

)

**Modalidades**

In [ ]:
pd.crosstab(
    df['modalidad_de_contratacion'],
    df['periodo_presidencial']
).plot(
    kind='bar',
    stacked=True
)

**Evolución temporal**

In [ ]:
df['anio'] = (
    pd.to_datetime(
        df['fecha_de_firma']
    ).dt.year
)

In [ ]:
sns.countplot(
    x='anio',
    data=df
)

**ANALÍTICA DIAGNÓSTICA**

Hallazgo 1

Concentración.

Ya tienes HHI.

Agregar:

In [ ]:
sns.histplot(
    hhi_resumen['HHI']
)

Hallazgo 2

Prórrogas.

In [ ]:
sns.boxplot(

data=df,

x='modalidad_de_contratacion',

y='desviacion_duracion'

)

Hallazgo 3

Comparación de perfiles

In [ ]:
perfil_entidad.groupby(
    'cluster_comprador'
).mean()

HALLAZGO 4

Relación anomalías vs riesgo

In [ ]:
pd.crosstab(

df['es_anomalo_iso'],

df['tiene_adicion_critica'],

normalize='index'

)

**MÉTODO DEL CODO**

Antes del K-Means

In [ ]:
inercias = []

for k in range(2,11):

    modelo = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )

    modelo.fit(
        X_perfil_scaled
    )

    inercias.append(
        modelo.inertia_
    )

In [ ]:
plt.plot(
    range(2,11),
    inercias,
    marker='o'
)

**ANALÍTICA PREDICTIVA**

In [ ]:
f2 = fbeta_score(
    y_test,
    y_pred_rf,
    beta=2
)

print(
    "F2 Score:",
    round(f2,4)
)

In [ ]:
La métrica principal es Recall y F2 Score.

En auditoría resulta más costoso
no detectar un contrato riesgoso
(falso negativo)
que revisar un contrato sano
(falso positivo).

**IMPORTANCIA DE VARIABLES**

In [ ]:
feature_names = (

pipe_rf
.named_steps['prep']
.get_feature_names_out()

)

In [ ]:
importancias = (

pipe_rf
.named_steps['clf']
.feature_importances_

)

In [ ]:
imp = pd.DataFrame({

    'variable':feature_names,
    'importancia':importancias

})

imp = imp.sort_values(
    'importancia',
    ascending=False
)

In [ ]:
plt.figure(figsize=(10,8))

sns.barplot(

data=imp.head(15),

x='importancia',

y='variable'

)

plt.title(
"Variables más importantes para predecir riesgo contractual"
)

**ANALÍTICA PRESCRIPTIVA**

In [ ]:
for corte in [0.50,0.60,0.70,0.80]:

    alertas = (
        y_prob_rf >= corte
    ).sum()

    print(
        corte,
        alertas
    )

In [ ]:
UMBRAL_CORTE = 0.60

In [ ]:
CONCLUSIÓN FINAL

No termines con AUC.

El análisis permitió caracterizar la contratación
pública colombiana durante los periodos
2018-2022 y 2022-2026.

Se identificaron patrones de concentración
de proveedores, perfiles diferenciados de entidades
y comportamientos atípicos mediante técnicas
de clustering y detección de anomalías.

Los modelos predictivos demostraron capacidad
para identificar contratos con riesgo
de desviación contractual.

Sin embargo, el valor principal del proyecto
no radica en la métrica predictiva obtenida,
sino en la posibilidad de transformar dichas
predicciones en decisiones de auditoría.

Se recomienda priorizar la revisión preventiva
de contratos cuya probabilidad estimada
supere el 60%, permitiendo focalizar recursos
limitados de supervisión sobre los casos
con mayor impacto potencial.